# Etapa 2: Selección de técnica de muestreo para la construcción de muestra inicial

**Materia:** Análisis de grandes volúmenes de datos (Gpo 10)  
**Institución:** Tecnológico de Monterrey, Posgrados  
**Equipo 12:**

- Carlos Eduardo Vega Campos (A01797803)
- Marco Emilio Jimenez Jimenez (A01797948)
- Martha Alicia Villalobos Facundo (A01840063)
- Jonathan Javier Monsalve Giraldo (A01840272)

**Profesores:** Dr. Iván Olmos Pineda, Luis Daniel Mendoza  
**Fecha:** 17 de mayo de 2026  
**Dataset:** NYC TLC Yellow Taxi Trip Records 2024-2025

## Objetivo del notebook

Construir una muestra representativa M de la población de viajes Yellow Taxi NYC mediante muestreo estratificado con calibración histórica. La Etapa 1 del proyecto caracterizó el dataset; esta etapa parte de los datos crudos, aplica limpieza basada en los hallazgos de Etapa 1, valida D contra distribuciones históricas verificables, y extrae M mediante `sampleBy` con piso mínimo por estrato. El notebook está pensado para ejecutarse de forma portable, tanto localmente como en Google Colab; todas las rutas de datos son relativas al notebook (`./data/raw`).

## 1. Configuración del entorno

Iniciamos sesión local de Spark. Configuración mínima: subir `spark.sql.debug.maxToStringFields` para evitar truncamiento de logs en agregaciones grandes.

Notebook portable: las rutas son relativas. Funciona en Ubuntu VM local y en Google Colab si previamente se instala Java.

In [ ]:
# Dependencias de Python para el notebook. Idempotente.
!pip install -q pyspark findspark pandas matplotlib

In [ ]:
# Solo en Google Colab: descomentar para instalar Java (la JVM que ejecuta Spark).
# Localmente con env-pyspark esta línea no es necesaria.
# !apt-get install openjdk-8-jdk-headless -qq > /dev/null

In [ ]:
import findspark
findspark.init()

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StructType, StructField, ByteType, ShortType,
    FloatType, TimestampNTZType, StringType,
)
from pathlib import Path
import json

spark = SparkSession.builder.master("local[*]").getOrCreate()

# Sube el umbral del log de planes (default 25) para evitar WARN benignos al agregar
# múltiples columnas en una sola llamada.
spark.conf.set("spark.sql.debug.maxToStringFields", 100)

print(f"Spark versión: {spark.version}")

### 1.2 Descarga reproducible de los datos

Reusamos el patrón de Etapa 1: descarga idempotente desde el CDN público de TLC a `./data/raw/` solo si el archivo no existe. Esto mantiene el notebook autosuficiente y portable. En una máquina donde ya se descargaron los 24 parquets mensuales más el catálogo de zonas (por ejemplo, tras ejecutar el notebook de Etapa 1), todas las descargas devuelven `skip` y la celda termina en segundos.

In [ ]:
import subprocess

CDN_BASE = "https://d37ci6vzurychx.cloudfront.net"
DATA_DIR = Path("data/raw")
YEARS = [2024, 2025]
LOOKUP_FILE = "taxi_zone_lookup.csv"


def download_if_missing(download_url, target_path):
    """Descarga `download_url` a `target_path` solo si `target_path` no existe.

    Devuelve un string con el estado: 'skip', 'ok' o 'error: <mensaje>'.
    """
    if target_path.exists():
        return "skip"

    target_path.parent.mkdir(parents=True, exist_ok=True)
    result = subprocess.run(
        ["curl", "-sSL", "-o", str(target_path), download_url],
        capture_output=True,
        timeout=900,
    )

    if result.returncode != 0:
        return f"error: curl exit {result.returncode}"

    return "ok"

In [ ]:
# Parquets mensuales de viajes
for year in YEARS:
    for month in range(1, 13):
        filename = f"yellow_tripdata_{year}-{month:02d}.parquet"
        url = f"{CDN_BASE}/trip-data/{filename}"
        target = DATA_DIR / filename
        status = download_if_missing(url, target)
        print(f"{status:>6}  {filename}")

# Tabla de referencia de zonas de taxi
url = f"{CDN_BASE}/misc/{LOOKUP_FILE}"
target = DATA_DIR / LOOKUP_FILE
status = download_if_missing(url, target)
print(f"{status:>6}  {LOOKUP_FILE}")

In [ ]:
files = sorted(DATA_DIR.glob("*"))
total_bytes = sum(f.stat().st_size for f in files)

for f in files:
    size_mb = f.stat().st_size / (1024 ** 2)
    print(f"{size_mb:>8.1f} MB   {f.name}")

print()
print(f"Archivos: {len(files)} (esperados: {len(YEARS) * 12 + 1})")
print(f"Tamaño total: {total_bytes / (1024 ** 3):.2f} GB")

## 2. Carga del dataset con esquema explícito

Reusamos el esquema validado en Etapa 1 en vez de inferirlo. El esquema con downcast (`tinyint` para enums, `smallint` para zonas, `float` para montos) ahorra del orden de 72 bytes por fila sobre los tipos por defecto (`long` para enums, `double` para montos), lo que equivale a aproximadamente 6 GB de presión de memoria sobre el dataset completo (Etapa 1, sección 6).

Mantenemos `mergeSchema=True` porque la columna `cbd_congestion_fee` solo aparece en archivos a partir de 2025-01-05. Sin esta opción, Spark tomaría el esquema del primer archivo y descartaría la columna en archivos posteriores. Con merge, los registros 2024 quedan con `cbd_congestion_fee = null` por diseño.

Referencia oficial: https://spark.apache.org/docs/latest/sql-data-sources-parquet.html#schema-merging

In [ ]:
YELLOW_SCHEMA = StructType([
    StructField("VendorID", ByteType(), True),
    StructField("tpep_pickup_datetime", TimestampNTZType(), True),
    StructField("tpep_dropoff_datetime", TimestampNTZType(), True),
    StructField("passenger_count", ByteType(), True),
    StructField("trip_distance", FloatType(), True),
    StructField("RatecodeID", ByteType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", ShortType(), True),
    StructField("DOLocationID", ShortType(), True),
    StructField("payment_type", ByteType(), True),
    StructField("fare_amount", FloatType(), True),
    StructField("extra", FloatType(), True),
    StructField("mta_tax", FloatType(), True),
    StructField("tip_amount", FloatType(), True),
    StructField("tolls_amount", FloatType(), True),
    StructField("improvement_surcharge", FloatType(), True),
    StructField("total_amount", FloatType(), True),
    StructField("congestion_surcharge", FloatType(), True),
    StructField("Airport_fee", FloatType(), True),
    StructField("cbd_congestion_fee", FloatType(), True),
])

print(f"Campos en el esquema: {len(YELLOW_SCHEMA.fields)}")

In [ ]:
parquet_paths = sorted(str(p) for p in DATA_DIR.glob("yellow_tripdata_*.parquet"))

df_raw = (spark.read
    .option("mergeSchema", "true")
    .schema(YELLOW_SCHEMA)
    .parquet(*parquet_paths))

zones = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(DATA_DIR / "taxi_zone_lookup.csv")))

print(f"Archivos parquet cargados: {len(parquet_paths)}")
print(f"Particiones del DataFrame de viajes: {df_raw.rdd.getNumPartitions()}")
print(f"Zonas en el catálogo: {zones.count()}")

In [ ]:
n_raw = df_raw.count()
print(f"Registros crudos en D: {n_raw:,}")

El conteo esperado es 89,892,322 registros, consistente con el total observado en Etapa 1 (sección 3) sobre los 24 archivos parquet mensuales. El catálogo de zonas debe traer 265 entradas. Cualquier diferencia indica que algún archivo no se descargó correctamente o que TLC publicó una revisión del dataset.